In [1]:
import pandas as pd 
import numpy as np

data = pd.read_csv("customers.csv")  # Loading the data

data.drop(columns=['customerID'], inplace=True) # Dropping uneccesary column
data.drop(columns=['gender'], inplace=True) # Dropping uneccesary column

data['TotalCharges'] = data['TotalCharges'].str.strip() # Removes whitespaces: " " -> ""
data.replace('', np.nan, inplace=True)    # Find any "" and replace into nan value

data.dropna(subset=['TotalCharges'], inplace=True) # Drop rows with nan values
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')



In [2]:

data['MultipleLines'] = data['MultipleLines'].replace({'No phone service': 'No'})
data['Churn'] = data['Churn'].replace({'Yes':1, 'No':0})

non_numeric_df = data.select_dtypes(exclude=['number'])

for col in non_numeric_df.columns:
    if data[col].nunique() == 3 and col != 'InternetService' and col != 'Contract':
        data[col] = data[col].replace({'No internet service': 'No'})


# Iterate through the filtered columns
for col in non_numeric_df.columns:
    if data[col].nunique() == 2 and col != 'Churn' :
        data[col] = data[col].replace({'Yes':1, 'No':0}).astype(int)

data = pd.get_dummies(data, dtype=int, drop_first=True)

data.info()


/tmp/ipykernel_34738/3332051837.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Churn'] = data['Churn'].replace({'Yes':1, 'No':0})
/tmp/ipykernel_34738/3332051837.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col] = data[col].replace({'Yes':1, 'No':0}).astype(int)
/tmp/ipykernel_34738/3332051837.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future be

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 23 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   SeniorCitizen                          7032 non-null   int64  
 1   Partner                                7032 non-null   int64  
 2   Dependents                             7032 non-null   int64  
 3   tenure                                 7032 non-null   int64  
 4   PhoneService                           7032 non-null   int64  
 5   MultipleLines                          7032 non-null   int64  
 6   OnlineSecurity                         7032 non-null   int64  
 7   OnlineBackup                           7032 non-null   int64  
 8   DeviceProtection                       7032 non-null   int64  
 9   TechSupport                            7032 non-null   int64  
 10  StreamingTV                            7032 non-null   int64  
 11  Streaming

/tmp/ipykernel_34738/3332051837.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col] = data[col].replace({'Yes':1, 'No':0}).astype(int)
/tmp/ipykernel_34738/3332051837.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[col] = data[col].replace({'Yes':1, 'No':0}).astype(int)
/tmp/ipykernel_34738/3332051837.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the futu

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

y = data["Churn"]
X = data.drop(columns=['Churn'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42,
    shuffle=True
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [4]:
import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score

model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    eval_metric='logloss'
)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
score = roc_auc_score( y_test, y_pred)
print(f"Accuracy: {accuracy_score(y_test, y_pred): .4f} ")
print( classification_report( y_test, y_pred))
print(score)

Accuracy:  0.7903 
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1033
           1       0.62      0.54      0.58       374

    accuracy                           0.79      1407
   macro avg       0.73      0.71      0.72      1407
weighted avg       0.78      0.79      0.79      1407

0.7113710132473301


In [5]:
import joblib

joblib.dump(model, "model.pkl" )
joblib.dump( scaler, "scaler.pkl" )


['scaler.pkl']